In [1]:
import pandas as pd

path = '/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/data/CTA_-_Ridership_-_Daily_Boarding_Totals_20251110.csv'

df = pd.read_csv(path, parse_dates=['service_date'])

df.columns = ['date', 'day_type','bus', 'rail', 'total']
df = df.sort_values(by='date').set_index('date')

df = df.drop('total', axis=1)
df = df.drop_duplicates()

import numpy as np

df['bus'] = df['bus'].str.replace(',','')
df['rail'] = df['rail'].str.replace(',','')

df['bus'] = df['bus'].astype(np.int64)
df['rail'] = df['rail'].astype(np.int64)

In [2]:
df.head(3)

,day_type,bus,rail
date,,,
2001-01-01,U,297192,126455
2001-01-02,W,780827,501952
2001-01-03,W,824923,536432


In [3]:
df_mulvar = (df[['bus','rail']]/1e6).copy()
df_mulvar['next_day_type'] = df['day_type'].shift(-1)
mulvar_train = df_mulvar.iloc[:-1,:]["2016-01":"2018-12"]
mulvar_valid = df_mulvar.iloc[:-1,:]["2019-01":"2019-05"]
mulvar_test =  df_mulvar.iloc[:-1,:]["2019-06":]

In [4]:
mulvar_train.head(3)

,bus,rail,next_day_type
date,,,
2016-01-01,0.303321,0.319835,A
2016-01-02,0.448859,0.365509,U
2016-01-03,0.340540,0.287661,W


In [7]:
from sklearn.preprocessing import OneHotEncoder

onehotencoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
onehotencoder.fit(mulvar_train[['next_day_type']])
mulvar_train_2 = onehotencoder.transform(mulvar_train[['next_day_type']])

In [8]:
mulvar_train[['next_day_type']]

,next_day_type
date,
2016-01-01,A
2016-01-02,U
2016-01-03,W
2016-01-04,W
2016-01-05,W
...,...
2018-12-27,W
2018-12-28,A
2018-12-29,U


In [9]:
mulvar_train_2

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       ...,
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.]])

In [10]:
mulvar_train_2 = pd.DataFrame(
    mulvar_train_2,
    columns = list(onehotencoder.get_feature_names_out()),
    index = mulvar_train.index
)

In [11]:
mulvar_train_2

,next_day_type_A,next_day_type_U,next_day_type_W
date,,,
2016-01-01,1.0,0.0,0.0
2016-01-02,0.0,1.0,0.0
2016-01-03,0.0,0.0,1.0
2016-01-04,0.0,0.0,1.0
2016-01-05,0.0,0.0,1.0
...,...,...,...
2018-12-27,0.0,0.0,1.0
2018-12-28,1.0,0.0,0.0
2018-12-29,0.0,1.0,0.0


In [12]:
mulvar_train.head(3)

,bus,rail,next_day_type
date,,,
2016-01-01,0.303321,0.319835,A
2016-01-02,0.448859,0.365509,U
2016-01-03,0.340540,0.287661,W


In [13]:
mulvar_train.loc[:,'next_day_type_A'] = mulvar_train_2['next_day_type_A']
mulvar_train.loc[:,'next_day_type_U'] = mulvar_train_2['next_day_type_U']
mulvar_train.loc[:,'next_day_type_W'] = mulvar_train_2['next_day_type_W']
mulvar_train = mulvar_train.reindex(columns=['bus','rail','next_day_type_A','next_day_type_U','next_day_type_W'])

In [14]:
mulvar_valid_2 = onehotencoder.transform(mulvar_valid[['next_day_type']])
mulvar_valid_2 = pd.DataFrame(
    mulvar_valid_2,
    columns=list(onehotencoder.get_feature_names_out()),
    index = mulvar_valid.index
)
mulvar_valid.loc[:,'next_day_type_A'] = mulvar_valid_2['next_day_type_A']
mulvar_valid.loc[:,'next_day_type_U'] = mulvar_valid_2['next_day_type_U']
mulvar_valid.loc[:,'next_day_type_W'] = mulvar_valid_2['next_day_type_W']

mulvar_valid = mulvar_valid.reindex(columns=['bus','rail','next_day_type_A','next_day_type_U','next_day_type_W'])

In [15]:
mulvar_test_2 = onehotencoder.transform(mulvar_test[['next_day_type']])
mulvar_test_2 = pd.DataFrame(
    mulvar_test_2,
    columns=list(onehotencoder.get_feature_names_out()),
    index = mulvar_test.index
)
mulvar_test.loc[:,'next_day_type_A'] = mulvar_test_2['next_day_type_A']
mulvar_test.loc[:,'next_day_type_U'] = mulvar_test_2['next_day_type_U']
mulvar_test.loc[:,'next_day_type_W'] = mulvar_test_2['next_day_type_W']

mulvar_test = mulvar_test.reindex(columns=['bus','rail','next_day_type_A','next_day_type_U','next_day_type_W'])

Building the chunks and dataset

In [16]:
seq_length = 56
chunk_size = seq_length + 1
mv_rail_train_chunks = [mulvar_train.iloc[i:i+chunk_size].values for i in range(mulvar_train.shape[0]-chunk_size+1)]
mv_rail_valid_chunks = [mulvar_valid.iloc[i:i+chunk_size].values for i in range(mulvar_valid.shape[0]-chunk_size+1)]
mv_rail_test_chunks = [mulvar_test.iloc[i:i+chunk_size].values for i in range(mulvar_test.shape[0]-chunk_size+1)]

In [19]:
import torch
from torch.utils.data import Dataset, DataLoader

In [20]:
mulvar_train.head(3)

,bus,rail,next_day_type_A,next_day_type_U,next_day_type_W
date,,,,,
2016-01-01,0.303321,0.319835,1.0,0.0,0.0
2016-01-02,0.448859,0.365509,0.0,1.0,0.0
2016-01-03,0.340540,0.287661,0.0,0.0,1.0


In [21]:
class TimeSeriesDataset2(Dataset):
    def __init__(self, chunks, seq_length):
        super().__init__()
        self.chunks = chunks
        self.seq_length = seq_length
    def __len__(self):
        return len(self.chunks)
    def __getitem__(self, index):
        chunk = self.chunks[index, :, :]
        data , label = chunk[:seq_length, :], chunk[seq_length, 1].to(torch.float32).item() # rail value is in column 1
        return data.to(torch.float32), label

In [22]:
train_mulvar_ds = TimeSeriesDataset2(torch.tensor(np.array(mv_rail_train_chunks),dtype=torch.float16),seq_length=56)
valid_mulvar_ds = TimeSeriesDataset2(torch.tensor(np.array(mv_rail_valid_chunks),dtype=torch.float16), seq_length=56)
test_mulvar_ds = TimeSeriesDataset2(torch.tensor(np.array(mv_rail_test_chunks),dtype=torch.float16), seq_length=56)

In [23]:
a, b = train_mulvar_ds[0]

In [24]:
a.shape

torch.Size([56, 5])

In [26]:
train_mulvar_dl = DataLoader(train_mulvar_ds, batch_size=64, shuffle=True)
valid_mulvar_dl = DataLoader(valid_mulvar_ds, batch_size=64)
test_mulvar_dl = DataLoader(test_mulvar_ds, batch_size=64)

In [28]:
import torch.nn as nn

torch.manual_seed(1)

device = torch.device("mps")

class SimpleRNN2(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, 16)
        self.linear2 = nn.Linear(16,8)
        self.linear3 = nn.Linear(8,1)
        self.relu = nn.ReLU()
    def forward(self, x):
        out, hn = self.rnn(x)
        mid = self.relu(self.linear(hn[-1,:,:]))
        mid2 = self.relu(self.linear2(mid))
        return self.relu(self.linear3(mid2))

simple_model2 = SimpleRNN2(5, 32)
simple_model2 = simple_model2.to(device)